# Legal Clause Risk Scorer: Fine-Tuning LegalBERT

This notebook is part of the **Legal Clause Risk Scorer** project. It handles the fine-tuning of the `nlpaueb/legal-bert-base-uncased` model on the **CUAD** dataset for risk classification (Low, Medium, High).

### 1. Setup and Installation

In [ ]:
!pip install -q transformers "datasets<3.0.0" accelerate scikit-learn

### 2. Load and Preprocess CUAD Dataset

In [ ]:
from datasets import load_dataset
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import evaluate
import numpy as np

# Risk Mapping (can be adjusted)
risk_mapping = {
    # High Risk (2) - Critical clauses that impact liability and competition
    'Cap on Liability': 2,
    'Uncapped Liability': 2,
    'Indemnification': 2,
    'IP Ownership Assignment': 2,
    'Change of Control': 2,
    'Exclusivity': 2,
    'Non-Compete': 2,
    'Most Favored Nation': 2,
    'Liquidated Damages': 2,
    'Price Restrictions': 2,
    'Revenue/Profit Sharing': 2,
    'Non-Solicit of Customers': 2,
    'No-Solicit of Employees': 2,
    'No-Hiring': 2,

    # Medium Risk (1) - Operationally significant clauses
    'Termination for Convenience': 1,
    'Termination for Cause': 1,
    'Audit Rights': 1,
    'Confidentiality': 1,
    'License Grant': 1,
    'Insurance': 1,
    'Minimum Commitment': 1,
    'Warranty': 1,
    'Non-Disparagement': 1,
    'Anti-Assignment': 1,
    'Competitive Restriction Exception': 1,
    'Rofr/Rofo/Rofn': 1,
    'Post-Termination Services': 1,

    # Low Risk (0) - Standard boilerplates and general info
    'Governing Law': 0,
    'Effective Date': 0,
    'Expiration Date': 0,
    'Renewal Term': 0,
    'Notice Period to Terminate Renewal': 0,
    'Document Name': 0,
    'Parties': 0,
    'Agreement Date': 0,
    'Covenant Not to Sue': 0,
    'Third Party Beneficiary': 0,
    'Force Majeure': 0,
    'Affiliate License-Licensor': 0,
    'Affiliate License-Licensee': 0,
    'Joint IP Ownership': 0,
    'Non-Transferable License': 0,
    'Unlimited/Irrevocable License': 0,
    'Volume Restriction': 0
}


def map_to_risk(question):
    for key in risk_mapping:
        if key.lower() in question.lower():
            return risk_mapping[key]
    return 0 # Default to Low

print("Loading dataset...")
dataset = load_dataset("theatticusproject/cuad-qa", trust_remote_code = True)

def prepare_data(example):
    processed = []
    for qa in example['qas']:
        risk = map_to_risk(qa['question'])
        for answer in qa['answers']:
            processed.append({
                'text': answer['text'],
                'label': risk
            })
    return processed

# Flatten the dataset for classification
train_processed = []
for item in dataset['train']:
    train_processed.extend(prepare_data(item))

test_processed = []
for item in dataset['test']:
    test_processed.extend(prepare_data(item))

train_df = pd.DataFrame(train_processed)
test_df = pd.DataFrame(test_processed)

from datasets import Dataset
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

### 3. Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

### 4. Training

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained("nlpaueb/legal-bert-base-uncased", num_labels=3)

metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric.compute(predictions=predictions, references=labels)['accuracy']
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")['f1']
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()

### 5. Save Model

In [ ]:
model.save_pretrained("legal_risk_model")
tokenizer.save_pretrained("legal_risk_model")

# Zip for downloading
!zip -r legal_risk_model.zip legal_risk_model